# 03 – Können LLMs Bilder verstehen?

**Lernziel:** Wir vergleichen ein **reines Textmodell** mit einem **bildfähigen Modell** und lernen, sichtbare Tatsachen von Vermutungen zu unterscheiden.

- **Llama 3.1 (8B)** verarbeitet in unserer Installation **nur Text**.
- **Gemma 3 (4B)** verarbeitet **Text und Bilder**.



## 1. Selbst beobachten – was ist tatsächlich auf dem Bild?

Seht euch das Bild an und notiert zuerst selbst drei **sichtbare** Merkmale.

![Gemaltes Porträt – National Gallery of Art](https://api.nga.gov/iiif/a2e6da57-3cd1-4235-b20e-95dcaefed6c8/full/!800,800/0/default.jpg)

**Frage:** Welche Aussagen lassen sich direkt aus dem Bild ableiten, welche wären bereits Interpretationen?


## Init – Bild für beide Modelle bereitstellen

Wir verwenden für beide Versuche **dasselbe Bild** und **dieselbe Frage**. Der folgende Code lädt das Beispielbild einmal und stellt eine gemeinsame Fragefunktion bereit.

Voraussetzung: Ollama läuft lokal, und das Bild ist online erreichbar.


In [ ]:
import base64
import requests
from openai import OpenAI, APIError
from IPython.display import display, Markdown

BILD_URL = "https://api.nga.gov/iiif/a2e6da57-3cd1-4235-b20e-95dcaefed6c8/full/!800,800/0/default.jpg"
TEXTMODELL = "llama3.1:8b-instruct-q4_K_M"
VISIONMODELL = "gemma3:4b"

bild = requests.get(BILD_URL, timeout=30)
bild.raise_for_status()
bild_url = "data:image/jpeg;base64," + base64.b64encode(bild.content).decode("ascii")
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama", timeout=180.0)

def bild_fragen(modell, frage):
    antwort = client.chat.completions.create(
        model=modell,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": frage},
            {"type": "image_url", "image_url": {"url": bild_url}},
        ]}],
        temperature=0.2,
        max_tokens=350,
    )
    return antwort.choices[0].message.content or "(Keine Textantwort)"


## 2. Textmodell – was passiert mit einer Bildanfrage?

Wir senden **Llama 3.1 dieselbe Bildanfrage**, die gleich danach Gemma 3 erhält.

**Beobachtet:** Ein Textmodell ohne Vision-Unterstützung kann das Bild nicht auswerten. Je nach Ollama-Version wird die Anfrage **abgewiesen** oder es entsteht **keine verwertbare Bildantwort**. Ein technischer Fehler ist hier ein Hinweis auf die fehlende Modellfähigkeit – nicht die gewünschte Bildbeschreibung.


In [ ]:
frage = "Was ist auf diesem Bild zu sehen? Beschreibe nur sichtbare Merkmale."

try:
    print("Llama 3.1 (nur Text):")
    print(bild_fragen(TEXTMODELL, frage))
except APIError as error:
    print("Llama 3.1 kann diese Bildanfrage nicht verarbeiten.")
    print("Ollama-Meldung:", str(error).splitlines()[0][:300])


## 3. Vision-Modell – kann Gemma 3 das Bild beschreiben?

Jetzt erhält **Gemma 3** dieselbe Frage mit demselben Bild.

**Vergleicht:** Welche Angaben könnt ihr direkt im Bild bestätigen? Gibt es Unklarheiten oder erfundene Details?


In [ ]:
print("Gemma 3 (Text + Bild):")
display(Markdown(bild_fragen(VISIONMODELL, frage)))


## 4. Falsche Voraussetzung – widerspricht das Modell?

Das gezeigte Bild ist **kein Sportwettkampf**. Wir stellen absichtlich eine falsche Frage, um zu prüfen, ob das Modell die Voraussetzung korrigiert oder eine passende Geschichte erfindet.


In [ ]:
display(Markdown(bild_fragen(
    VISIONMODELL,
    "Welche beiden Sportteams spielen in diesem Bild gegeneinander?"
)))


## 5. Präziser Auftrag – Beobachtungen von Vermutungen trennen

Ein guter Bildauftrag kann das Risiko unbegründeter Aussagen senken. Wir bitten nun ausdrücklich um die Trennung von Beobachtung und Interpretation.


In [ ]:
GUTE_FRAGE = """Beschreibe das Bild auf Deutsch.
1. Nenne fünf Merkmale, die direkt sichtbar sind.
2. Nenne höchstens zwei Interpretationen und kennzeichne sie als Vermutung.
3. Sage ausdrücklich, was sich aus dem Bild nicht sicher bestimmen lässt.
"""

display(Markdown(bild_fragen(VISIONMODELL, GUTE_FRAGE)))


## Auswertung – wann ist Bildverständnis nützlich?

| Versuch | Was prüfen wir? |
|---|---|
| Llama 3.1 mit Bild | Textmodell ohne Vision-Unterstützung |
| Gemma 3 mit Bild | Erkennung sichtbarer Merkmale |
| Gemma 3 mit falscher Frage | Umgang mit falschen Voraussetzungen |
| Gemma 3 mit präzisem Auftrag | Trennung von Beobachtung und Vermutung |

**Erkenntnis:** Nicht jedes LLM kann Bilder verarbeiten. Ein Vision-Modell liefert Bildbeschreibungen, ist aber nicht automatisch zuverlässig. **Bildinhalt, Vermutung und belegte Aussage** müssen getrennt werden.

**Transfer:** Formuliert selbst eine Frage, die ausschliesslich anhand des Bildes beantwortet werden kann.

**Referenz:** [Gemma 3 in Ollama](https://ollama.com/library/gemma3), [Ollama – OpenAI-kompatible API](https://docs.ollama.com/api/openai-compatibility).
